# Unofficial [LeetArxiv](https://leetarxiv.substack.com/p/hnp-sum) Implementation of Hidden Number Problem with Small Unknown Multipliers (HNP-SUM)

HNP-SUM is the challenge of recovering small unknown multipliers (not unknown keys) using a lattice approach. We code the paper in Python.

<div align="center">
<img src="https://github.com/MurageKibicho/Hidden-Number-Problem-with-Small-Unknown-Multipliers/blob/main/Images/Abstract.png?raw=true:" alt="HNP-SUM Paper Abstract" width=400>
</div>

HNP-SUM is unlike other HNPs because we only know the observation. We don't know the multipliers, errors or secret key.

This is part of our series on _Practical Hidden Number Problems for Programmers_:

This code is best followed using this [LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).

*These links are paywalled. (It’s how I’ll afford grad school lol)

[Part 1: Quantum Computing and Intro to Hidden Number Problems
](https://leetarxiv.substack.com/p/linear-hidden-number-problem-zero-to-hero-for-computer-scientiests)

[Part 2: Increasing Lattice Volume for Improved HNP Attacks](https://leetarxiv.substack.com/p/guessing-bits-improved-lattice-attacks)

[Part 3: Hidden Number Problems with Multiple Noise Holes
](https://leetarxiv.substack.com/p/pythonsage-extended-hidden-number)

[Part 4: Hidden Number Problems with Chosen Multipliers](https://leetarxiv.substack.com/p/acgs-algorithm-for-hidden-number)

[Part 5: Hidden Number Problems with Chosen Errors](https://leetarxiv.substack.com/p/acgs-algorithm-for-hidden-number)

[Part 6 : Fourier Analysis Attacks on HNPs: Bleichenbacher's Bias](https://leetarxiv.substack.com/p/bleichenbacher-attacks-hnp)

**Part 7 (we are here)**: [Hidden Number Problems with Small Unknown Multipliers](https://leetarxiv.substack.com/p/hnp-sum).

## Install Libraries
This is [section 2.0 of the LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).
We need:
1. **fpylll** for lattice construction.
2. **cysignals** to interact with fpylll's C++ code

In [72]:
#Install fpylll for lattices
!pip install fpylll
!pip install cysignals

## Theorem 1: Testing for Solution Existence

This is [section 2.1 of the LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).

In [73]:
import math
def TestSolutionExistence(T, E, N, numberOfSamples):
  exponent = (numberOfSamples + 1) / (numberOfSamples - 1)
  result = (T ** exponent) * E
  return result, result < N

if __name__ == "__main__":
  N0 = 13441
  N1 = 205115282021455665897114700593932402728804164701536103180137503955397371
  testCases = [
    {"T": 1,  "E": N0//4, "N": N0, "samples": 3},
    {"T": 10, "E": N0//2, "N": N0, "samples": 10},
    {"T": 1,  "E": N1//4, "N": N1, "samples": 3},
    {"T": 4, "E": N1//4, "N": N1, "samples": 10},
  ]

  for i, case in enumerate(testCases):
    T = case["T"]
    E = case["E"]
    N = case["N"]
    samples = case["samples"]

    result, exists = TestSolutionExistence(T, E, N, samples)
    resultLog = math.log(result, 2)
    NLog = math.log(N, 2)
    print(f"Test {i}:")
    print(f"  T = {T}, N = {N}, samples = {samples}")
    print(f"  Result = {resultLog:.3f} < {NLog:.3f} : {exists}")


Test 0:
  T = 1, N = 13441, samples = 3
  Result = 11.714 < 13.714 : True
Test 1:
  T = 10, N = 13441, samples = 10
  Result = 16.774 < 13.714 : False
Test 2:
  T = 1, N = 205115282021455665897114700593932402728804164701536103180137503955397371, samples = 3
  Result = 234.893 < 236.893 : True
Test 3:
  T = 4, N = 205115282021455665897114700593932402728804164701536103180137503955397371, samples = 10
  Result = 237.338 < 236.893 : False


## Definition 1: HNP-SUM Dataset Generator
This is [section 2.2 of the LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).

Where $T$ and $E$ are known size bounds for the multiplier and error term, and $N$ is our field characteristic HNP-SUM is defined:
<div align="center">
<img src="https://github.com/MurageKibicho/Hidden-Number-Problem-with-Small-Unknown-Multipliers/blob/main/Images/HNP-SUM_Definition.png?raw=true:" alt="HNP-SUM definition" width=400>
</div>


In [74]:
import random
def HNP_SUM_Dataset(numberOfSamples, secretKey, multiplierBound, errorBound, fieldCharacteristic):
  x = secretKey
  T = multiplierBound
  E = errorBound
  N = fieldCharacteristic
  n = numberOfSamples
  observations = [];multipliers = [];errors = [];
  for i in range(0, n):
    t = random.randint(-T, T)
    e = random.randint(-E, E)
    a = (t * x + e) % N
    observations.append(a)
    multipliers.append(t)
    errors.append(e)
  return observations, multipliers, errors

if __name__ == "__main__":
  fieldCharacteristic = 13441
  secretKey = 1234
  multiplierBound = 10
  errorBound = 100
  numberOfSamples = 5

  observations,_,_ = HNP_SUM_Dataset(numberOfSamples, secretKey, multiplierBound, errorBound, fieldCharacteristic)
  print(f"Observations: {observations}")

Observations: [9905, 8572, 10926, 12316, 6101]


## Lattice Construction for HNP-SUM

This is [section 2.3 of the LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).

We solve HNP-SUM using the basis:

$$
B = \begin{pmatrix}
E & 0 & \cdots & 0 & a_1 \\
0 & E & \cdots & 0 & a_2 \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & \cdots & E & a_n \\
0 & 0 & \cdots & 0 & N
\end{pmatrix}
$$

---

### 3 Sample Example
When $E = 1$ and $n=3$, the basis and its solution resembles:
$$
\underbrace{
\begin{bmatrix}
t_2 & -t_1 & 0 & k_1 \\
t_3 & 0 & -t_1 & k_2 \\
0 & -t_3 & t_2 & k_3
\end{bmatrix}
}_{\text{Unknown Coefficient Matrix } C}
\cdot
\underbrace{
\begin{bmatrix}
1 & 0 & 0 & a_1 \\
0 & 1 & 0 & a_2 \\
0 & 0 & 1 & a_3 \\
0 & 0 & 0 & N
\end{bmatrix}
}_{\text{Known Basis Matrix } B}
=
\underbrace{
\begin{bmatrix}
t_2 & -t_1 & 0 & t_2e_1 - t_1e_2 \\
t_3 & 0 & -t_1 & t_3e_1 - t_1e_3 \\
0 & -t_3 & t_2 & t_3e_2 - t_2e_3
\end{bmatrix}
}_{\text{Short Target Vectors } V}
$$

---



In [75]:
from fpylll import FPLLL
from fpylll import *
from fpylll.util import gaussian_heuristic
from decimal import Decimal
from time import time
def HNP_SUM_ConstructLattice(observations, errorBound, fieldCharacteristic):
  latticeDimension = len(observations) + 1
  lattice = IntegerMatrix(latticeDimension, latticeDimension)
  #Set bottom right
  lattice[latticeDimension - 1, latticeDimension - 1] = int(fieldCharacteristic)
  for i in range(latticeDimension - 1):
    #Set error bound identity
    lattice[i, i] = int(errorBound)
    #Set observation column
    lattice[i, latticeDimension - 1] = int(observations[i])
  return lattice

if __name__ == "__main__":
  fieldCharacteristic = 13441
  secretKey = 1234
  multiplierBound = 10
  errorBound = 100
  numberOfSamples = 5

  observations,_,_  = HNP_SUM_Dataset(numberOfSamples, secretKey, multiplierBound, errorBound, fieldCharacteristic)
  lattice = HNP_SUM_ConstructLattice(observations, errorBound, fieldCharacteristic)
  print(f"{lattice}")

[ 100   0   0   0   0  8486 ]
[   0 100   0   0   0  2502 ]
[   0   0 100   0   0  3619 ]
[   0   0   0 100   0  5016 ]
[   0   0   0   0 100  9724 ]
[   0   0   0   0   0 13441 ]


## Testing code, small p
We extract the sub-lattice then find HNF in Sage math as seen in [section 2.4 of the LeetArxiv guide](https://leetarxiv.substack.com/p/hnp-sum).

In [93]:
from fpylll import IntegerMatrix
import random
import numpy as np
def ExportToSage(M):
  print("\nM = Matrix(ZZ,[")
  for i in range(M.nrows):
    row = [int(M[i, j]) for j in range(M.ncols)]
    print(f"  {row},")
  print("])")

def ExtractSublattice(lattice):
  """Extract the sublattice L(B_sub) of rank n-1 i.e the first n-1 vectors after LLL reduction"""
  latticeDimension = lattice.nrows #Assumes lattice is a square matrix
  subLattice = IntegerMatrix(latticeDimension - 2, latticeDimension)
  for i in range(latticeDimension - 2):
    for j in range(latticeDimension):
        subLattice[i, j] = lattice[i, j]
  return subLattice

def SolveHNPSUM_uptoSubLattice(observations, errorBound, fieldCharacteristic):
  lattice = HNP_SUM_ConstructLattice(observations, errorBound, fieldCharacteristic)
  print(f"Before Reduction:\n{lattice}")
  LLL.reduction(lattice)
  print(f"After Reduction:\n{lattice}")
  subLattice = ExtractSublattice(lattice)
  #print(f"SubLattice:\n{subLattice}")
  return subLattice


if __name__ == "__main__":
  fieldCharacteristic = 27006979257190
  observations = [16434376644250, 18067839662587, 6420926526082]
  subLattice = SolveHNPSUM_uptoSubLattice(observations, 1, fieldCharacteristic)
  ExportToSage(subLattice)
  print("\nPass the sub-lattice to Sage Math. Sympy's HNF is weird")

Before Reduction:
[ 1 0 0 16434376644250 ]
[ 0 1 0 18067839662587 ]
[ 0 0 1  6420926526082 ]
[ 0 0 0 27006979257190 ]
After Reduction:
[    -15    -25    12  -71 ]
[     47     66   -20  -68 ]
[  36967 -16082 25946 2238 ]
[ -12565  30656 63041 2494 ]

M = Matrix(ZZ,[
  [-15, -25, 12, -71],
  [47, 66, -20, -68],
])

Pass the sub-lattice to Sage Math. Sympy's HNF is weird


## Cryptographic-sized example

In [107]:

if __name__ == "__main__":
  random.seed(412)
  fieldCharacteristic = 205115282021455665897114700593932402728804164701536103180137503955397371
  secretKey = random.randint(2**134, 2**135)
  multiplierBound = 32
  errorBound = 2**100
  numberOfSamples = 5
  result, exists = TestSolutionExistence(multiplierBound, errorBound, fieldCharacteristic, samples)
  resultLog = math.log(result, 2);NLog = math.log(fieldCharacteristic, 2)
  print(f"  Result = {resultLog:.3f} < {NLog:.3f} : {exists}")

  observations,multipliers,_  = HNP_SUM_Dataset(numberOfSamples, secretKey, multiplierBound, errorBound, fieldCharacteristic)
  subLattice = SolveHNPSUM_uptoSubLattice(observations, 1, fieldCharacteristic)
  ExportToSage(subLattice)
  print(f"Expected: {multipliers}")

  Result = 106.111 < 236.893 : True
Before Reduction:
[ 1 0 0 0 0 205115282021455665897114700593468261312226097862092441843340517877563153 ]
[ 0 1 0 0 0 205115282021455665897114700593736974763930231628825982110088609781387856 ]
[ 0 0 1 0 0 205115282021455665897114700593150690869303506246050938628626042048475726 ]
[ 0 0 0 1 0                               317570442922182406227523718447585826840951 ]
[ 0 0 0 0 1                                97713982438522105205015908250692128200410 ]
[ 0 0 0 0 0 205115282021455665897114700593932402728804164701536103180137503955397371 ]
After Reduction:
[                      20735063                     -27810961                        6066691                       20223581                       25676517  -6481013 ]
[                      56609234                     -20539641                      -49316771                      -51934742                        2068323 -18483814 ]
[                     -68715500                     -27218201            

# Modified HNP-SUM

Start from
$$
\begin{aligned}
e_0 &\equiv t_0 x + a_0 \pmod N \\
e_1 &\equiv t_1 x + a_1 \pmod N
\end{aligned}
$$

where $x$ is the unknown, $N$ is the modulus, and $t_i$, $a_i$, and $e_i$ are the corresponding values for the $i$th equation.

It follows
$$
\begin{aligned}
t_1 e_0 - t_0 e_1
&\equiv t_1(t_0x+a_0)-t_0(t_1x+a_1) \pmod N \\
&\equiv t_1a_0-t_0a_1 \pmod N.
\end{aligned}
$$

Equivalently,

$$
e_0t_1-e_1t_0 \equiv t_1a_0-t_0a_1 \pmod N.
$$

Our lattice in the three sample case is
$$
\underbrace{
\begin{bmatrix}
e_1 & -e_0 & 0 & k_0 \\
e_2 & 0 & -e_0 & k_1 \\
0 & -e_2 & e_1 & k_2
\end{bmatrix}
}_{\text{Unknown Coefficient Matrix } C}
\cdot
\underbrace{
\begin{bmatrix}
1 & 0 & 0 & t_0 \\
0 & 1 & 0 & t_1 \\
0 & 0 & 1 & t_2 \\
0 & 0 & 0 & N
\end{bmatrix}
}_{\text{Known Basis Matrix } B}
=
\underbrace{
\begin{bmatrix}
e_1 & -e_0 & 0 & t_1a_0 - t_0a_1 \\
e_2 & 0 & -e_0 & t_2a_0 - t_0a_2 \\
0 & -e_2 & e_1 & t_2a_1 - t_1a_2
\end{bmatrix}
}_{\text{Short Target Vectors } V}
$$